# 05 — Credits enrichment & talent-aware modeling

**Phase:** Metadata + tabular engineering (notebooks 03–04) plateaued in explainable signal. This notebook **expands the business data contract** with **people / talent / lightweight IP** signals from TMDB credits — still strictly **pre-release** (credits are known at greenlight in this PoC).

**Workflow:** why talent data matters → load & merge → join diagnostics → concise credits EDA → **leakage-safe** frequency features (fit on **train only**) → same modeling stack as notebook 04 → **three-way** comparison (baseline vs engineered vs credits) → interpretation tied to **actual metrics** → business conclusions → roadmap.

**Style:** **Observation → Business interpretation → Modeling implication** (same rhythm as notebooks 01–04).

**Non-goals:** post-release proxies in `X`, opaque feature soup, or “algorithm magic” without a data story.


## 1. Business objective — why credits matter

Studios rarely greenlight on **budget and genre alone**. Distribution and production teams anchor narratives on **talent concentration** (marquee cast, director track record), **production ecosystem scale** (crew depth), and **familiar IP / sequel positioning** (even crude text cues).

The TMDB credits extract gives us a **structured cast/crew roster** before release — a realistic next step when tabular metadata stops explaining outcomes.

We still frame the task as **ranking / triage support**: many drivers (marketing intensity, competition, true star wattage beyond name lists) remain **unobserved**.

---

**Observation:** Outcomes correlate weakly with “spreadsheet metadata” alone in many entertainment PoCs.  
**Business interpretation:** Talent and packaging are how executives **reduce uncertainty** before cameras roll.  
**Modeling implication:** Any frequency-based talent signal must be **fit on training data only**, then applied to validation/test — otherwise we fake performance with **split leakage**.


## 2. Load datasets

- **Modeling base:** `data/processed/movies_cleaned_with_target.csv` (same contract as notebooks 03–04).  
- **Credits extension:** `data/raw/tmdb_5000_credits.csv` (`movie_id`, `title`, `cast`, `crew` as JSON-like strings).

`PROJECT_ROOT` resolves whether the notebook runs from repo root or `notebooks/`.

---

**Observation:** Two-file workflows are normal in analytics — joins are where **schema drift** appears.  
**Business interpretation:** Credits quality is part of the **risk register** for model consumers.  
**Modeling implication:** Print **merge diagnostics** before trusting new columns.


In [1]:
from __future__ import annotations

import ast
import json
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

_CWD = Path.cwd().resolve()
if (_CWD / "data" / "processed").is_dir():
    PROJECT_ROOT = _CWD
elif (_CWD.parent / "data" / "processed").is_dir():
    PROJECT_ROOT = _CWD.parent
else:
    PROJECT_ROOT = _CWD
    print("Warning: data/processed not found; using cwd as PROJECT_ROOT.")

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "movies_cleaned_with_target.csv"
CREDITS_PATH = PROJECT_ROOT / "data" / "raw" / "tmdb_5000_credits.csv"
PLOTS_DIR = PROJECT_ROOT / "plots" / "modeling"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="talk", font_scale=0.95)
plt.rcParams["figure.figsize"] = (11, 5.5)
plt.rcParams["axes.titlesize"] = 14


def save_fig(name: str) -> Path:
    path = PLOTS_DIR / f"{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor="white", edgecolor="none")
    plt.close()
    return path


TARGET_COLUMN = "movie_success_class"
FORBIDDEN_IN_X = {"revenue", "roi", "log_roi", TARGET_COLUMN}
LEAKAGE_DROP = ["revenue", "roi", "log_roi", TARGET_COLUMN]

BASELINE_CANDIDATES = [
    "budget",
    "runtime",
    "main_genre",
    "original_language",
    "release_month",
    "release_quarter",
    "genre_count",
    "production_company_count",
    "production_country_count",
    "spoken_language_count",
]

STATIC_ENGINEERED = [
    "budget_log",
    "runtime_bucket",
    "international_production",
    "multilingual_movie",
    "release_season",
    "genre_complexity",
    "decade",
]

ENGINEERED_FOR_MODEL = STATIC_ENGINEERED + ["production_scale"]

CREDITS_NUMERIC = [
    "director_movie_count",
    "top_director_flag",
    "top_billed_cast_count",
    "known_actor_count",
    "cast_size",
    "crew_size",
    "writer_count",
    "possible_franchise_flag",
    "ensemble_cast_flag",
    "talent_score",
]
CREDITS_CATEGORICAL = ["director_bucket"]

TOP_DIRECTORS_K = 40
TOP_ACTORS_K = 120
DIRECTOR_BUCKET_TOP_N = 25

df = None
df_credits = None
df_join = None

if not DATA_PATH.exists():
    print("Missing processed movies:", DATA_PATH)
else:
    df = pd.read_csv(DATA_PATH)
    print("Loaded movies:", DATA_PATH, "| shape:", df.shape)
    print("TARGET_COLUMN:", TARGET_COLUMN)
    print("Leakage columns in movies file:", [c for c in LEAKAGE_DROP if c in df.columns])

if not CREDITS_PATH.exists():
    print("Missing credits file:", CREDITS_PATH)
else:
    df_credits = pd.read_csv(CREDITS_PATH)
    print("Loaded credits:", CREDITS_PATH, "| shape:", df_credits.shape)


Loaded movies: /Users/paulbensemoun/Documents/ml_poc/new-project/data/processed/movies_cleaned_with_target.csv | shape: (3229, 30)
TARGET_COLUMN: movie_success_class
Leakage columns in movies file: ['revenue', 'roi', 'movie_success_class']
Loaded credits: /Users/paulbensemoun/Documents/ml_poc/new-project/data/raw/tmdb_5000_credits.csv | shape: (4803, 4)


## 3. Merge strategy & join audit

**Primary key:** `movies_cleaned_with_target.id` ↔ `tmdb_5000_credits.movie_id` (TMDB movie identifier).

We use a **left join** from the processed modeling base to credits so we never drop rows from the target file. We print **row counts, unmatched rate, and duplicate-key checks** — joins are the #1 place for silent data loss.

**Join risks (explicit):** title strings can differ between files; we do **not** match on title. If `movie_id` drifts relative to `id` in a future export, match rate will drop — the diagnostics below surface that immediately.

---

**Observation:** One-to-many credit rows per movie would break naive merges.  
**Business interpretation:** Data engineering hygiene is part of **model risk disclosure**.  
**Modeling implication:** Fail loudly in diagnostics rather than silently padding `NaN` everywhere.


In [2]:
if df is None:
    print("Skip merge.")
else:
    n_before = len(df)
    n_id_dup = int(df["id"].duplicated().sum())
    print("\n===== MERGE DIAGNOSTICS =====")
    print("Movies rows:", n_before, "| duplicate `id`:", n_id_dup)

    if df_credits is not None:
        n_cred = len(df_credits)
        n_mid_dup = int(df_credits["movie_id"].duplicated().sum())
        print("Credits rows:", n_cred, "| duplicate `movie_id`:", n_mid_dup)

        try:
            df_join = df.merge(
                df_credits,
                left_on="id",
                right_on="movie_id",
                how="left",
                suffixes=("", "_cred"),
                validate="one_to_one",
            )
        except Exception as exc:
            print("merge validate one_to_one failed, retrying without validate:", exc)
            df_join = df.merge(
                df_credits,
                left_on="id",
                right_on="movie_id",
                how="left",
                suffixes=("", "_cred"),
            )
        n_after = len(df_join)
        print("Rows after left merge:", n_after)
        unmatched = int(df_join["cast"].isna().sum())
        print("Movies without credits match (missing cast):", unmatched, f"({100 * unmatched / max(n_after,1):.2f}%)")
        if "movie_id" in df_join.columns:
            df_join = df_join.drop(columns=["movie_id"])
        extra_titles = [c for c in df_join.columns if c in ("title_cred", "title_credits")]
        for c in extra_titles:
            if c in df_join.columns:
                df_join = df_join.drop(columns=[c])
    else:
        df_join = df.copy()
        print("Credits missing — df_join = movies only.")

    print("Final joined columns (count):", len(df_join.columns))



===== MERGE DIAGNOSTICS =====
Movies rows: 3229 | duplicate `id`: 0
Credits rows: 4803 | duplicate `movie_id`: 0
Rows after left merge: 3229
Movies without credits match (missing cast): 0 (0.00%)
Final joined columns (count): 32


## 4. Credits exploration (concise, business-oriented)

We quantify **roster size**, **crew depth**, and **director concentration** — enough to sanity-check parsing without turning this notebook into a pure EDA report.

---

**Observation:** Long-tail cast lists are normal; billing order matters more than tail extras for “star packaging.”  
**Business interpretation:** Cast count alone is a **coarse** proxy for ensemble vs intimate pictures.  
**Modeling implication:** Pair raw sizes with **train-only “known talent”** counts to separate volume from reputation.


In [3]:
def safe_json_list(val) -> list:
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return []
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        s = val.strip()
        if not s:
            return []
        try:
            return json.loads(s)
        except Exception:
            try:
                return ast.literal_eval(s)
            except Exception:
                return []
    return []


def extract_director_name(crew_raw) -> str:
    for job in ("Director", "Co-Director"):
        for c in safe_json_list(crew_raw):
            if not isinstance(c, dict):
                continue
            if str(c.get("job", "")) == job:
                n = c.get("name")
                if n:
                    return str(n).strip()
    return "__missing__"


def _order_key(x: dict) -> int:
    o = x.get("order", 999)
    try:
        return int(o)
    except (TypeError, ValueError):
        return 999


def extract_cast_ordered(cast_raw) -> list[dict]:
    lst = [x for x in safe_json_list(cast_raw) if isinstance(x, dict)]
    lst.sort(key=_order_key)
    return lst


def writer_count_from_crew(crew_raw) -> int:
    n = 0
    for c in safe_json_list(crew_raw):
        if not isinstance(c, dict):
            continue
        dep = str(c.get("department", ""))
        job = str(c.get("job", "")).lower()
        if dep == "Writing" or "writer" in job:
            n += 1
    return n


_FRANCHISE_PAT = re.compile(
    r"\b(part|chapter|returns|begins|rise|sequel|prequel|reloaded|revolutions|legacy)\b|"
    r"\b(ii|iii|iv|v|vi|vii|viii|ix)\b|"
    r"\bvs\.?\b|"
    r"\s2$|\s3$|:?\s2\b|:?\s3\b|\b2:\s|\b3:\s",
    re.I,
)


def franchise_heuristic(title, keywords_raw) -> int:
    parts = []
    if title is not None and not (isinstance(title, float) and pd.isna(title)):
        parts.append(str(title))
    for kw in safe_json_list(keywords_raw):
        if isinstance(kw, dict) and kw.get("name"):
            parts.append(str(kw["name"]))
    blob = " ".join(parts).lower()
    return int(bool(_FRANCHISE_PAT.search(blob)))


if df_join is None:
    print("Skip parsing.")
else:
    directors = []
    cast_sizes = []
    crew_sizes = []
    writers = []
    top_billed = []
    franch = []
    ensemble = []

    for _, row in df_join.iterrows():
        cr = row.get("crew")
        ca = row.get("cast")
        directors.append(extract_director_name(cr))
        cl = extract_cast_ordered(ca)
        cast_sizes.append(len(cl))
        crew_sizes.append(len(safe_json_list(cr)))
        writers.append(writer_count_from_crew(cr))
        billed = 0
        for ent in cl[:5]:
            if ent.get("name"):
                billed += 1
        top_billed.append(billed)
        kw = row.get("keywords")
        franch.append(franchise_heuristic(row.get("title"), kw))
        ensemble.append(int(len(cl) >= 12))

    df_join = df_join.copy()
    df_join["director_name"] = directors
    df_join["cast_size"] = cast_sizes
    df_join["crew_size"] = crew_sizes
    df_join["writer_count"] = writers
    df_join["top_billed_cast_count"] = top_billed
    df_join["possible_franchise_flag"] = franch
    df_join["ensemble_cast_flag"] = ensemble

    print("\n===== CREDITS SUMMARY STATS =====")
    print("cast_size: mean=%.1f median=%.0f max=%d" % (df_join["cast_size"].mean(), df_join["cast_size"].median(), df_join["cast_size"].max()))
    print("crew_size: mean=%.1f median=%.0f max=%d" % (df_join["crew_size"].mean(), df_join["crew_size"].median(), df_join["crew_size"].max()))
    print("writer_count: mean=%.2f" % (df_join["writer_count"].mean()))
    print("possible_franchise_flag rate:", round(df_join["possible_franchise_flag"].mean(), 3))
    print("Top 12 directors by movie count (global preview, not used for modeling):")
    print(df_join["director_name"].value_counts().head(12))

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    sns.histplot(df_join["cast_size"], bins=30, kde=False, ax=axes[0], color="#2c5282")
    axes[0].set_title("Cast list length")
    sns.histplot(df_join["crew_size"], bins=30, kde=False, ax=axes[1], color="#276749")
    axes[1].set_title("Crew list length")
    save_fig("07_credits_cast_crew_histograms")

    miss_dir = (df_join["director_name"] == "__missing__").mean()
    print("Share missing director:", round(float(miss_dir), 4))



===== CREDITS SUMMARY STATS =====
cast_size: mean=26.0 median=19 max=224
crew_size: mean=33.9 median=20 max=435
writer_count: mean=2.49
possible_franchise_flag rate: 0.103
Top 12 directors by movie count (global preview, not used for modeling):
director_name
Steven Spielberg     27
Clint Eastwood       19
Martin Scorsese      16
Robert Rodriguez     16
Ridley Scott         16
Renny Harlin         15
Tim Burton           14
Steven Soderbergh    14
Robert Zemeckis      13
Oliver Stone         13
Tony Scott           12
Michael Bay          12
Name: count, dtype: int64
Share missing director: 0.0006


## 5. Tabular features from notebook 04 (same definitions)

We rebuild **static** engineered columns on the joined frame. `production_scale` stays **`__pending__`** until after the split (train-only tertiles, identical logic to notebook 04).

---

**Observation:** Iteration 04 features remain valid on the wider dataframe.  
**Business interpretation:** The credits experiment **adds a layer**, it does not rewrite the tabular story.  
**Modeling implication:** Three-way comparison isolates **talent incremental value** over tabular engineering.


In [4]:
if df_join is None:
    print("Skip tabular engineering.")
else:
    d = df_join
    d["budget_log"] = np.log1p(d["budget"].clip(lower=0))

    def _runtime_bucket(v) -> str:
        if pd.isna(v):
            return "__missing__"
        r = float(v)
        if r < 90:
            return "short"
        if r <= 120:
            return "medium"
        return "long"

    d["runtime_bucket"] = d["runtime"].map(_runtime_bucket)
    d["international_production"] = (d["production_country_count"].fillna(0) > 1).astype(int)
    d["multilingual_movie"] = (d["spoken_language_count"].fillna(0) > 1).astype(int)

    def _season_from_month(m) -> str:
        if pd.isna(m):
            return "__missing__"
        mi = int(m)
        if mi in (12, 1, 2):
            return "winter"
        if mi in (3, 4, 5):
            return "spring"
        if mi in (6, 7, 8):
            return "summer"
        if mi in (9, 10, 11):
            return "fall"
        return "__missing__"

    if "release_month" in d.columns:
        d["release_season"] = d["release_month"].map(_season_from_month)
    else:
        d["release_season"] = "__missing__"

    def _genre_complexity(gc) -> str:
        if pd.isna(gc):
            return "__missing__"
        g = int(gc)
        if g <= 1:
            return "focused"
        if g <= 3:
            return "mixed"
        return "hybrid"

    d["genre_complexity"] = d["genre_count"].map(_genre_complexity)

    if "release_year" in d.columns:
        yr = pd.to_numeric(d["release_year"], errors="coerce")

        def _decade(y):
            if pd.isna(y):
                return "__missing__"
            yi = int(y)
            return f"{(yi // 10) * 10}s"

        d["decade"] = yr.map(_decade)
    else:
        d["decade"] = "__missing__"

    d["production_scale"] = "__pending__"
    df_join = d
    print("Tabular engineered columns attached. Shape:", df_join.shape)


Tabular engineered columns attached. Shape: (3229, 47)


## 6. Leakage discipline — train-only talent frequencies

**Why this matters:** using **full-corpus** director or actor frequencies to label train and test rows leaks **future co-occurrence** information into early splits — a classic way to **inflate** validation metrics in entertainment datasets.

**What we do instead:**

1. `train_test_split(..., stratify=y, random_state=42)` — identical to notebooks 03–04.  
2. Build **director movie counts**, **top-director membership**, **actor “fame” sets**, **`director_bucket` (top-N names)**, and **`talent_score` scaling** using **training rows only**.  
3. Map those dictionaries / scalers onto **train and test** separately.

**Naming note:** `director_name` is retained in the analytical frame for QA; the model consumes **`director_bucket`** — train-top-`DIRECTOR_BUCKET_TOP_N` names plus **`__other__`** — to keep one-hot cardinality presentation-safe.

`FORBIDDEN_IN_X` columns never enter `X`. After all columns exist, we **assert** the contract again.

---

**Observation:** Joins widen the attack surface for subtle leakage (global popularity).  
**Business interpretation:** Defensible talent features are **historical relative to the training portfolio**, not omniscient.  
**Modeling implication:** If you later move to **time-based splits**, re-fit these dictionaries per training era.


In [5]:
def production_scale_from_train(train_counts: pd.Series, apply_counts: pd.Series) -> pd.Series:
    t = pd.to_numeric(train_counts, errors="coerce").dropna()
    if t.empty:
        q1, q2 = 0.0, 1.0
    else:
        q1, q2 = t.quantile(1 / 3), t.quantile(2 / 3)
    if q1 == q2:
        q2 = q2 + 1e-6

    def bucket(x):
        if pd.isna(x):
            return "__missing__"
        xv = float(x)
        if xv <= q1:
            return "indie"
        if xv <= q2:
            return "mid_scale"
        return "large_scale"

    return apply_counts.map(bucket)


def apply_train_only_talent_features(df_all: pd.DataFrame, idx_train: np.ndarray, idx_test: np.ndarray) -> pd.DataFrame:
    out = df_all.copy()
    tr = out.iloc[idx_train]
    te = out.iloc[idx_test]

    # production_scale (uses production_company_count from baseline columns)
    tr_co = tr["production_company_count"]
    out.loc[tr.index, "production_scale"] = production_scale_from_train(tr_co, tr["production_company_count"]).astype(str)
    out.loc[te.index, "production_scale"] = production_scale_from_train(tr_co, te["production_company_count"]).astype(str)

    # director frequencies on TRAIN only
    dir_counts = tr.loc[tr["director_name"] != "__missing__", "director_name"].value_counts()
    top_directors = set(dir_counts.nlargest(TOP_DIRECTORS_K).index)

    actor_counts: dict[str, int] = {}
    for _, row in tr.iterrows():
        cl = extract_cast_ordered(row.get("cast"))
        seen = set()
        for ent in cl[:12]:
            nm = ent.get("name")
            if not nm:
                continue
            nm = str(nm).strip()
            if nm in seen:
                continue
            seen.add(nm)
            actor_counts[nm] = actor_counts.get(nm, 0) + 1
    top_actors = set(sorted(actor_counts, key=lambda k: actor_counts[k], reverse=True)[:TOP_ACTORS_K])

    dir_movie_count_map = dir_counts.to_dict()
    top_dir_names = set(dir_counts.nlargest(DIRECTOR_BUCKET_TOP_N).index)

    def map_dir_count(name):
        if name == "__missing__":
            return 0
        return int(dir_movie_count_map.get(name, 0))

    def map_top_flag(name):
        return int(name in top_directors and name != "__missing__")

    def map_bucket(name):
        if name == "__missing__":
            return "__missing__"
        if name in top_dir_names:
            return name
        return "__other__"

    def map_known_actor(cast_raw) -> int:
        k = 0
        cl = extract_cast_ordered(cast_raw)
        seen = set()
        for ent in cl[:12]:
            nm = ent.get("name")
            if not nm:
                continue
            nm = str(nm).strip()
            if nm in seen:
                continue
            seen.add(nm)
            if nm in top_actors:
                k += 1
        return k

    out["director_movie_count"] = out["director_name"].map(map_dir_count)
    out["top_director_flag"] = out["director_name"].map(map_top_flag)
    out["director_bucket"] = out["director_name"].map(map_bucket)
    out["known_actor_count"] = out["cast"].map(map_known_actor)

    # talent_score: min-max on train, then weighted blend (interpretable, not tuned)
    mm = MinMaxScaler()
    tr_x = tr.assign(_ka=out.loc[tr.index, "known_actor_count"], _dm=out.loc[tr.index, "director_movie_count"])[["_ka", "_dm"]].astype(float)
    mm.fit(tr_x)
    both = out.assign(_ka=out["known_actor_count"], _dm=out["director_movie_count"])[["_ka", "_dm"]].astype(float)
    scaled = mm.transform(both)
    top_flag = out["top_director_flag"].astype(float).values
    out["talent_score"] = 0.45 * scaled[:, 0] + 0.35 * scaled[:, 1] + 0.20 * top_flag
    return out


if df_join is None:
    df_model = None
    idx_train = idx_test = None
    print("Skip train-only feature build.")
else:
    y_all = df_join[TARGET_COLUMN].astype(str)
    idx = np.arange(len(df_join))
    idx_train, idx_test = train_test_split(
        idx,
        test_size=0.2,
        random_state=42,
        stratify=y_all,
    )
    df_model = apply_train_only_talent_features(df_join, idx_train, idx_test)

    present_base = [c for c in BASELINE_CANDIDATES if c in df_model.columns]
    BASELINE_FEATURES = [c for c in present_base if c not in FORBIDDEN_IN_X]
    ENRICHED_FEATURES = BASELINE_FEATURES + [c for c in ENGINEERED_FOR_MODEL if c in df_model.columns]
    CREDITS_FEATURES = ENRICHED_FEATURES + [c for c in CREDITS_NUMERIC + CREDITS_CATEGORICAL if c in df_model.columns]

    print("\n===== LEAKAGE AUDIT (FEATURE LISTS) =====")
    print("FORBIDDEN_IN_X:", sorted(FORBIDDEN_IN_X))
    for label, cols in [
        ("baseline", BASELINE_FEATURES),
        ("engineered", ENRICHED_FEATURES),
        ("credits", CREDITS_FEATURES),
    ]:
        leak = set(cols) & FORBIDDEN_IN_X
        assert not leak, leak
        assert TARGET_COLUMN not in cols
        print(f"n_features {label}:", len(cols))

    print("\nTrain rows:", len(idx_train), "| Test rows:", len(idx_test))



===== LEAKAGE AUDIT (FEATURE LISTS) =====
FORBIDDEN_IN_X: ['log_roi', 'movie_success_class', 'revenue', 'roi']
n_features baseline: 10
n_features engineered: 18
n_features credits: 29

Train rows: 2583 | Test rows: 646


## 7. Modeling — three comparable feature regimes

**Regimes**

1. **Baseline** — notebook 03 tabular contract.  
2. **Engineered** — baseline + notebook 04 features (including train-only `production_scale`).  
3. **Credits** — engineered + talent / IP / roster features (all train-safe as above).

**Models & protocol:** same as notebook 04 — `LogisticRegression` (balanced), `RandomForestClassifier` (balanced), `GradientBoostingClassifier`, optional `XGBClassifier`; **macro-F1** + accuracy on the **shared** stratified holdout.

---

**Observation:** Three regimes tell you whether talent data **adds incremental lift** beyond tabular engineering.  
**Business interpretation:** Negative deltas are informative — they discipline scope and data spend.  
**Modeling implication:** Keep `random_state=42` so results are reproducible in reviews.


In [6]:
HAS_XGB = False
try:
    from xgboost import XGBClassifier

    HAS_XGB = True
except Exception:
    XGBClassifier = None
    print("xgboost not available — skipping XGBClassifier.")


def prepare_X(feature_cols: list[str], frame: pd.DataFrame) -> tuple[pd.DataFrame, list[str], list[str]]:
    X = frame[feature_cols].copy()
    baseline_cat = ["main_genre", "original_language", "release_month", "release_quarter"]
    extra_cat = ["runtime_bucket", "production_scale", "release_season", "genre_complexity", "decade"]
    credits_cat = list(CREDITS_CATEGORICAL)
    cat_cols = [c for c in baseline_cat + extra_cat + credits_cat if c in X.columns]
    num_cols = [c for c in feature_cols if c not in cat_cols]

    for col in ["main_genre", "original_language"]:
        if col in X.columns:
            X[col] = X[col].astype("string").fillna("__missing__")
    for col in ["release_month", "release_quarter"]:
        if col in X.columns:
            X[col] = X[col].apply(lambda v: "__missing__" if pd.isna(v) else str(int(v)))
    for col in cat_cols:
        X[col] = X[col].astype(str)
    for col in cat_cols:
        nu = X[col].nunique(dropna=False)
        if nu > 200:
            print(f"WARNING: `{col}` has {nu} unique values (>200).")

    return X, num_cols, cat_cols


def build_preprocessor(num_cols: list[str], cat_cols: list[str]) -> ColumnTransformer:
    num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
    cat_pipe = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )
    tr = []
    if num_cols:
        tr.append(("num", num_pipe, num_cols))
    if cat_cols:
        tr.append(("cat", cat_pipe, cat_cols))
    return ColumnTransformer(transformers=tr)


def train_suite(X_train, X_test, y_train, y_test, num_cols, cat_cols, banner: str):
    def mk_prep():
        return build_preprocessor(num_cols, cat_cols)

    def mk_lr():
        return Pipeline(
            [
                ("prep", mk_prep()),
                (
                    "clf",
                    LogisticRegression(
                        max_iter=3000,
                        class_weight="balanced",
                        random_state=42,
                        solver="lbfgs",
                    ),
                ),
            ]
        )

    def mk_rf():
        return Pipeline(
            [
                ("prep", mk_prep()),
                (
                    "clf",
                    RandomForestClassifier(
                        n_estimators=200,
                        class_weight="balanced",
                        random_state=42,
                        n_jobs=-1,
                    ),
                ),
            ]
        )

    def mk_gbc():
        return Pipeline([("prep", mk_prep()), ("clf", GradientBoostingClassifier(random_state=42))])

    builders = {"logistic_regression": mk_lr, "random_forest": mk_rf, "gradient_boosting": mk_gbc}
    if HAS_XGB:

        def mk_xgb():
            return Pipeline(
                [
                    ("prep", mk_prep()),
                    (
                        "clf",
                        XGBClassifier(
                            n_estimators=200,
                            max_depth=5,
                            learning_rate=0.1,
                            objective="multi:softprob",
                            num_class=int(y_train.nunique()),
                            random_state=42,
                            n_jobs=-1,
                            eval_metric="mlogloss",
                        ),
                    ),
                ]
            )

        builders["xgboost"] = mk_xgb

    print("\n===== " + banner + " =====")
    rows = []
    models = {}
    for name, mk in builders.items():
        pipe = mk()
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        rows.append(
            {
                "model": name,
                "accuracy": accuracy_score(y_test, pred),
                "macro_f1": f1_score(y_test, pred, average="macro"),
            }
        )
        models[name] = pipe
        print(name, "| acc", round(rows[-1]["accuracy"], 3), "| macro_f1", round(rows[-1]["macro_f1"], 3))
    return pd.DataFrame(rows), models


if df_model is None:
    triple_comparison = None
    fitted_credits = {}
    y_train = y_test = None
    X_test_credits = None
    print("Skip modeling.")
else:
    y = df_model[TARGET_COLUMN].astype(str)
    y_train = y.iloc[idx_train]
    y_test = y.iloc[idx_test]

    def slice_X(cols):
        X_full, nc, cc = prepare_X(cols, df_model)
        return X_full.iloc[idx_train], X_full.iloc[idx_test], nc, cc

    Xtr_b, Xte_b, nb, cb = slice_X(BASELINE_FEATURES)
    Xtr_e, Xte_e, ne, ce = slice_X(ENRICHED_FEATURES)
    Xtr_c, Xte_c, ncr, ccr = slice_X(CREDITS_FEATURES)

    print("\n===== SHAPES =====")
    print("baseline train/test:", Xtr_b.shape, Xte_b.shape)
    print("engineered train/test:", Xtr_e.shape, Xte_e.shape)
    print("credits train/test:", Xtr_c.shape, Xte_c.shape)

    rb, fb = train_suite(Xtr_b, Xte_b, y_train, y_test, nb, cb, "MODEL TRAINING — BASELINE")
    re, fe = train_suite(Xtr_e, Xte_e, y_train, y_test, ne, ce, "MODEL TRAINING — ENGINEERED")
    rc, fitted_credits = train_suite(Xtr_c, Xte_c, y_train, y_test, ncr, ccr, "MODEL TRAINING — CREDITS")

    triple_comparison = rb.rename(
        columns={"macro_f1": "baseline_macro_f1", "accuracy": "baseline_accuracy"}
    ).merge(re.rename(columns={"macro_f1": "engineered_macro_f1", "accuracy": "engineered_accuracy"}), on="model").merge(
        rc.rename(columns={"macro_f1": "credits_macro_f1", "accuracy": "credits_accuracy"}), on="model"
    )
    triple_comparison["delta_eng_vs_base"] = (
        triple_comparison["engineered_macro_f1"] - triple_comparison["baseline_macro_f1"]
    )
    triple_comparison["delta_cred_vs_base"] = (
        triple_comparison["credits_macro_f1"] - triple_comparison["baseline_macro_f1"]
    )
    triple_comparison["delta_cred_vs_eng"] = (
        triple_comparison["credits_macro_f1"] - triple_comparison["engineered_macro_f1"]
    )
    triple_comparison = triple_comparison.sort_values("credits_macro_f1", ascending=False).reset_index(drop=True)

    print("\n===== THREE-WAY COMPARISON (macro-F1 on test) =====")
    display(
        triple_comparison[
            [
                "model",
                "baseline_macro_f1",
                "engineered_macro_f1",
                "credits_macro_f1",
                "delta_eng_vs_base",
                "delta_cred_vs_base",
                "delta_cred_vs_eng",
            ]
        ].round(3)
    )

    X_test_credits = Xte_c


xgboost not available — skipping XGBClassifier.

===== SHAPES =====
baseline train/test: (2583, 10) (646, 10)
engineered train/test: (2583, 18) (646, 18)
credits train/test: (2583, 29) (646, 29)

===== MODEL TRAINING — BASELINE =====
logistic_regression | acc 0.393 | macro_f1 0.373
random_forest | acc 0.533 | macro_f1 0.332
gradient_boosting | acc 0.551 | macro_f1 0.305

===== MODEL TRAINING — ENGINEERED =====
logistic_regression | acc 0.42 | macro_f1 0.39
random_forest | acc 0.526 | macro_f1 0.306
gradient_boosting | acc 0.54 | macro_f1 0.291

===== MODEL TRAINING — CREDITS =====
logistic_regression | acc 0.472 | macro_f1 0.438
random_forest | acc 0.59 | macro_f1 0.377
gradient_boosting | acc 0.584 | macro_f1 0.404

===== THREE-WAY COMPARISON (macro-F1 on test) =====


,model,baseline_macro_f1,engineered_macro_f1,credits_macro_f1,delta_eng_vs_base,delta_cred_vs_base,delta_cred_vs_eng
0,logistic_regression,0.373,0.390,0.438,0.017,0.065,0.048
1,gradient_boosting,0.305,0.291,0.404,-0.014,0.099,0.113
2,random_forest,0.332,0.306,0.377,-0.026,0.045,0.071


## 8. Interpretation — read the table honestly

The next cell prints **auto-generated, metric-aware commentary** (still written in consulting language). It never invents numbers — it reacts to the **empirical** `triple_comparison` dataframe from your run.

---

**Observation:** Narratives should follow evidence, especially when lifts are within noise.  
**Business interpretation:** Mixed results still inform **where to invest next** (data vs modeling).  
**Modeling implication:** Pair metrics with **confusion matrices** and **importance views** before changing the roadmap.


In [7]:
if triple_comparison is None:
    print("Skip interpretation block.")
else:
    best = triple_comparison.iloc[0]
    print("\n===== AUTO INTERPRETATION (THIS RUN) =====")
    print("Best credits model by macro-F1:", best["model"])
    print(
        "Mean delta credits vs baseline (macro-F1):",
        round(triple_comparison["delta_cred_vs_base"].mean(), 4),
    )
    print(
        "Mean delta credits vs engineered (macro-F1):",
        round(triple_comparison["delta_cred_vs_eng"].mean(), 4),
    )

    lr = triple_comparison[triple_comparison["model"] == "logistic_regression"]
    if not lr.empty:
        lr = lr.iloc[0]
        if lr["delta_cred_vs_base"] > 0.01:
            print(
                "\n• Logistic regression gained meaningfully vs baseline with credits features — "
                "linear models sometimes benefit from **bounded numeric talent scores** and low-cardinality buckets."
            )
        elif lr["delta_cred_vs_base"] < -0.005:
            print(
                "\n• Logistic regression did not improve (or worsened) — **collinear talent proxies** or **sparse OHE** may hurt the linear separable approximation."
            )
        else:
            print("\n• Logistic regression: **small/no change** — talent signals may be weakly linear or need stronger regularization / feature selection.")

    trees = triple_comparison[triple_comparison["model"].isin(["random_forest", "gradient_boosting", "xgboost"])]
    if not trees.empty:
        if trees["delta_cred_vs_base"].median() > 0.01:
            print(
                "\n• Tree models median uplift vs baseline suggests some **non-linear** use of roster / director structure."
            )
        elif trees["delta_cred_vs_base"].median() < -0.01:
            print(
                "\n• Tree models median **down** vs baseline — possible **noise absorption** or **variance** from wider sparse design; do not over-interpret single splits."
            )
        else:
            print("\n• Tree models: **flat vs baseline** — credits may not add stable splits at this granularity.")

    if triple_comparison["delta_cred_vs_eng"].max() < 0.005:
        print(
            "\n**Headline:** Credits features did **not** materially beat tabular engineering on this split — "
            "a credible outcome; next leverage is often **external marketing/IP data** or **time-based validation**."
        )
    else:
        print(
            "\n**Headline:** At least one model shows a **non-trivial** lift from credits vs engineered alone — "
            "worth a deeper **error analysis** (which titles moved, which classes gained recall)."
        )



===== AUTO INTERPRETATION (THIS RUN) =====
Best credits model by macro-F1: logistic_regression
Mean delta credits vs baseline (macro-F1): 0.0697
Mean delta credits vs engineered (macro-F1): 0.0773

• Logistic regression gained meaningfully vs baseline with credits features — linear models sometimes benefit from **bounded numeric talent scores** and low-cardinality buckets.

• Tree models median uplift vs baseline suggests some **non-linear** use of roster / director structure.

**Headline:** At least one model shows a **non-trivial** lift from credits vs engineered alone — worth a deeper **error analysis** (which titles moved, which classes gained recall).


## 9. Figures — macro-F1, confusion matrix, importances

Saved under `plots/modeling/` with the same hygiene as notebook 04 (`tight_layout`, white background, `plt.close`).

---

**Observation:** Visuals anchor executive conversations when tables feel abstract.  
**Business interpretation:** Confusion matrices show **where** the model spends errors (often majority-class comfort).  
**Modeling implication:** Importance on trees is **descriptive**, not causal — use it to **hypothesize**, not to mandate.


In [8]:
if triple_comparison is None or not len(fitted_credits):
    print("Skip figures.")
else:
    long_df = triple_comparison.melt(
        id_vars="model",
        value_vars=["baseline_macro_f1", "engineered_macro_f1", "credits_macro_f1"],
        var_name="regime",
        value_name="macro_f1",
    )
    long_df["regime"] = long_df["regime"].map(
        {
            "baseline_macro_f1": "baseline",
            "engineered_macro_f1": "engineered",
            "credits_macro_f1": "credits",
        }
    )
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(data=long_df, x="model", y="macro_f1", hue="regime", palette="deep", ax=ax)
    ax.set_title("Macro-F1: baseline vs engineered vs credits (test)")
    ax.set_ylim(0, 1)
    plt.xticks(rotation=20, ha="right")
    save_fig("08_credits_three_way_macrof1")

    best_name = triple_comparison.iloc[0]["model"]
    pipe = fitted_credits[best_name]
    y_hat = pipe.predict(X_test_credits)
    print("\nBest credits model:", best_name)
    print(classification_report(y_test, y_hat, digits=3))
    pref = ["flop", "average", "hit"]
    labels = [c for c in pref if c in set(y_test) | set(y_hat)]
    labels += sorted((set(y_test) | set(y_hat)) - set(labels))
    cm = confusion_matrix(y_test, y_hat, labels=labels)
    fig, ax = plt.subplots(figsize=(7, 5.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(f"Confusion matrix — credits — {best_name}")
    save_fig("09_credits_confusion_best")

    print("\n===== FEATURE IMPORTANCE (BEST TREE, CREDITS REGIME) =====")
    tree_names = [m for m in ["random_forest", "gradient_boosting", "xgboost"] if m in fitted_credits]
    tree_rank = triple_comparison[triple_comparison["model"].isin(tree_names)].sort_values(
        "credits_macro_f1", ascending=False
    )
    talent_tokens = (
        "director",
        "known_actor",
        "cast_size",
        "crew",
        "writer",
        "franchise",
        "ensemble",
        "talent_score",
    )
    if tree_rank.empty:
        print("No tree models — skip importance.")
    else:
        bt = tree_rank.iloc[0]["model"]
        tp = fitted_credits[bt]
        clf, prep = tp.named_steps["clf"], tp.named_steps["prep"]
        try:
            names = prep.get_feature_names_out()
            if hasattr(clf, "feature_importances_") and len(names) == len(clf.feature_importances_):
                fi = pd.Series(clf.feature_importances_, index=names).sort_values(ascending=False)
                top = fi.head(22)
                fig, ax = plt.subplots(figsize=(9, 7.5))
                ax.barh(top.index.astype(str), top.values, color="#553c9a", edgecolor="white", linewidth=0.5)
                ax.invert_yaxis()
                ax.set_title(f"Top importances — credits — {bt}")
                ax.set_xlabel("Importance")
                save_fig("10_credits_feature_importance_top")

                sub_idx = [i for i in fi.index if any(t in str(i) for t in talent_tokens)]
                sub = fi.loc[sub_idx].sort_values(ascending=False).head(12)
                if len(sub):
                    fig, ax = plt.subplots(figsize=(8, 5))
                    ax.barh(sub.index.astype(str), sub.values, color="#805ad5", edgecolor="white", linewidth=0.5)
                    ax.invert_yaxis()
                    ax.set_title("Talent / IP–related importances (subset)")
                    ax.set_xlabel("Importance")
                    save_fig("11_credits_talent_importance_subset")
                display(top.to_frame("importance").round(4))
            else:
                print("Warning: could not align importances with feature names.")
        except Exception as exc:
            print("Warning: importance extraction failed:", exc)



Best credits model: logistic_regression
              precision    recall  f1-score   support

     average      0.247     0.325     0.281       126
        flop      0.376     0.582     0.457       158
         hit      0.732     0.475     0.576       362

    accuracy                          0.472       646
   macro avg      0.451     0.461     0.438       646
weighted avg      0.550     0.472     0.489       646


===== FEATURE IMPORTANCE (BEST TREE, CREDITS REGIME) =====


,importance
num__cast_size,0.1599
num__crew_size,0.0925
num__budget,0.0823
num__budget_log,0.0769
num__director_movie_count,0.0617
num__talent_score,0.0503
num__possible_franchise_flag,0.0366
num__runtime,0.0336
num__production_company_count,0.0332
num__writer_count,0.0302


## 10. Business conclusions

**Talent data is necessary but not sufficient** in real studio workflows: names on a call sheet do not encode **marketing spend**, **franchise strength beyond title keywords**, or **release competition**.

What this notebook *does* establish is a **professional pattern**: expand the data contract, **re-audit leakage**, compare regimes on the **same split**, and interpret results **without hype**.

If credits features **barely move** macro-F1, the honest client message is: “We need richer **pre-release commercial** signals, not more trees.” If they **do move** the needle, the next step is **targeted error analysis** and **time-based validation** — not immediate production deployment.

---

**Observation:** Incremental value is often uneven across model families.  
**Business interpretation:** Executives care about **stable ranking**, not a single accuracy point.  
**Modeling implication:** Calibrated probabilities and cost-sensitive thresholds come **after** we trust the evaluation protocol.


## 11. Next improvements (roadmap)

- **Text / NLP:** synopsis TF–IDF or embeddings (governance + interpretability plan).  
- **External data:** critic aggregates, social buzz proxies, marketing estimates (hard to source, high value).  
- **Time-based validation:** train older / validate newer to stress-test **era drift**.  
- **Calibration:** `CalibratedClassifierCV` for decisions that read as probabilities.  
- **Ensembling:** stacking only after **honest** OOF generation to avoid leakage.  
- **Entity graph features:** recurring creative teams (writer–director pairs) with strict train-only counts.

---

**Observation:** The best roadmaps alternate **new information** with **evaluation hygiene**.  
**Business interpretation:** Each backlog item should map to a **portfolio decision**.  
**Modeling implication:** Version datasets whenever joins or dictionaries change materially.
